# Smart MCQ Solver — Pretrained Transformer 5-Fold Cross-Validation

**Model type :** 5-Fold Stratified Cross-Validation Fine-Tuning  
**Base model :** `google/electra-base-discriminator` (~110 M parameters)  
**Target score:** ≥ 0.80+ MAP@3 on Kaggle Public Leaderboard  
**Key Highlights:**
1. **`[CLS]` Token Representation Pooling** — Optimized for transformer encoder classification
2. **Standard AdamW (lr=2e-5)** — Smooth gradient flow without learning rate starvation
3. **5-Fold Stratified Cross Validation** — Covers entire dataset & eliminates single-split variance
4. **Full Sequence Context (max_length = 256)** — Un-truncated question and option tokens
5. **Test-Time Out-of-Fold Averaging** — Ensembles all 5 fold models for final test predictions

## 1. Environment & Dependencies

> **Kaggle users:** run this cell:
> ```
> !pip install -q transformers>=4.40.0 accelerate>=0.28.0 datasets>=2.18.0 wandb sentencepiece scikit-learn
> ```
>
> **Local users:** the cell below uses `uv pip install`.

In [ ]:
import subprocess, sys, os

PACKAGES = [
    "transformers>=4.40.0",
    "accelerate>=0.28.0",
    "datasets>=2.18.0",
    "wandb",
    "sentencepiece",
    "scikit-learn",
]

if not os.path.exists("/kaggle"):
    print("Installing via uv...")
    subprocess.check_call(["uv", "pip", "install", *PACKAGES])
    print("Dependencies ready ✓")
else:
    print("Kaggle detected — skip this cell and use !pip install instead.")

## 2. Imports, Seed & Global Configuration

In [ ]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import wandb
from tqdm.auto import tqdm

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


seed_everything(SEED)

# ── Device ───────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Paths (Kaggle vs Local) ───────────────────────────────────────
if os.path.exists("/kaggle"):
    TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
    TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
    OUTPUT_DIR = "/kaggle/working"
else:
    TRAIN_PATH = "../Data/train.csv"
    TEST_PATH = "../Data/test.csv"
    OUTPUT_DIR = "../Models/electra_5fold"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Model & Hyperparameters ───────────────────────────────────────
OPTIONS = ["A", "B", "C", "D", "E"]
MODEL_NAME = "google/electra-base-discriminator"
WANDB_PROJECT = "24F2008474-t22026"

N_FOLDS = 5
EPOCHS = 5
BATCH_SIZE = 8  # Per device
ACCUM_STEPS = 4  # Effective batch size = 32
MAX_LEN = 256
LEARNING_RATE = 2e-5

print(f"Device       : {DEVICE}")
if torch.cuda.is_available():
    print(
        f"GPU          : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)"
    )
print(f"PyTorch      : {torch.__version__}")
print(f"Base Model   : {MODEL_NAME}")

## 3. W&B Login & MAP@3 Metric Function

In [ ]:
wandb.login(
    key="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"
)


def mapk(actual: list, predicted: list, k: int = 3) -> float:
    scores = []
    for act, preds in zip(actual, predicted):
        ap = 0.0
        for rank, pred in enumerate(preds[:k], start=1):
            if pred == act:
                ap = 1.0 / rank
                break
        scores.append(ap)
    return float(np.mean(scores))


def compute_metrics(logits: np.ndarray, true_answer_letters: list) -> dict:
    y_true = [OPTIONS.index(ans) for ans in true_answer_letters]
    y_pred = np.argmax(logits, axis=1)
    preds_top3 = [
        [OPTIONS[j] for j in np.argsort(-logits[k])[:3]] for k in range(len(logits))
    ]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "map3": mapk(true_answer_letters, preds_top3),
    }


assert abs(mapk(["A"], [["A", "B", "C"]]) - 1.0) < 1e-9
assert abs(mapk(["A"], [["B", "A", "C"]]) - 0.5) < 1e-9
assert abs(mapk(["A"], [["C", "D", "A"]]) - 1 / 3) < 1e-9
print("MAP@3 functions verified ✓")

## 4. Tokenizer & Dataset Class

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}  |  Vocab Size: {tokenizer.vocab_size:,}")


class MCQDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        max_length: int = MAX_LEN,
        is_test: bool = False,
    ):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        input_ids, attention_mask = [], []
        prompt_text = str(row["prompt"])

        for opt in OPTIONS:
            option_text = str(row[opt])
            enc = self.tokenizer(
                prompt_text,
                option_text,
                max_length=self.max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            input_ids.append(enc["input_ids"].squeeze(0))
            attention_mask.append(enc["attention_mask"].squeeze(0))

        input_ids = torch.stack(input_ids)  # (5, max_length)
        attention_mask = torch.stack(attention_mask)  # (5, max_length)

        if self.is_test:
            return input_ids, attention_mask

        target = torch.tensor(OPTIONS.index(row["answer"]), dtype=torch.long)
        return input_ids, attention_mask, target

## 5. Model Architecture (`[CLS]` Representation Classifier)

In [ ]:
class ElectraMCQClassifier(nn.Module):
    def __init__(self, model_name: str = MODEL_NAME, dropout_rate: float = 0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor
    ) -> torch.Tensor:
        b, n, l = input_ids.size()
        flat_ids = input_ids.view(b * n, l)
        flat_mask = attention_mask.view(b * n, l)

        outputs = self.encoder(input_ids=flat_ids, attention_mask=flat_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :]  # [CLS] token representation
        pooled = self.dropout(cls_rep)
        logits = self.classifier(pooled).view(b, n)  # (batch_size, 5)
        return logits


model_test = ElectraMCQClassifier().to(DEVICE)
print(
    f"Electra MCQ Classifier loaded ✓  |  Total parameters: {sum(p.numel() for p in model_test.parameters()) / 1e6:.1f} M"
)
del model_test
torch.cuda.empty_cache()

## 6. 5-Fold Stratified Cross-Validation Execution

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
n_train, n_test = len(train_df), len(test_df)

run = wandb.init(
    project=WANDB_PROJECT,
    name="electra-5fold-cv",
    config={
        "model": MODEL_NAME,
        "folds": N_FOLDS,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE * ACCUM_STEPS,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LEN,
    },
    tags=["electra", "5fold-cv", "cls-pooling"],
    reinit="finish_previous",
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_oof_logits = np.zeros((n_train, 5))
test_logits_matrix = np.zeros((n_test, 5))

test_ds = MCQDataset(test_df, tokenizer, max_length=MAX_LEN, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\nStarting 5-Fold Cross-Validation ({MODEL_NAME})...")
print("=" * 65)

for fold, (train_idx, val_idx) in enumerate(
    skf.split(train_df, train_df["answer"]), start=1
):
    print(f"\n>>> FOLD {fold}/{N_FOLDS} <<<")
    train_fold_df = train_df.iloc[train_idx]
    val_fold_df = train_df.iloc[val_idx]

    train_ds_fold = MCQDataset(train_fold_df, tokenizer, max_length=MAX_LEN)
    val_ds_fold = MCQDataset(val_fold_df, tokenizer, max_length=MAX_LEN)

    train_loader = DataLoader(
        train_ds_fold, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        val_ds_fold, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    model = ElectraMCQClassifier(MODEL_NAME).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=0.01
    )
    criterion = nn.CrossEntropyLoss()

    total_steps = (len(train_loader) // ACCUM_STEPS) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps,
    )

    best_val_map3 = 0.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()

        for step, (input_ids, attention_mask, targets) in enumerate(
            tqdm(train_loader, desc=f"Fold {fold} | Epoch {epoch}/{EPOCHS}")
        ):
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            targets = targets.to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, targets) / ACCUM_STEPS
            loss.backward()
            running_loss += loss.item() * ACCUM_STEPS

            if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        # ── Validation Evaluation ──────────────────────────────────
        model.eval()
        val_logits_list = []
        with torch.no_grad():
            for input_ids, attention_mask, _ in val_loader:
                val_logits_list.append(
                    model(input_ids.to(DEVICE), attention_mask.to(DEVICE)).cpu().numpy()
                )

        val_logits_arr = np.concatenate(val_logits_list, axis=0)
        fold_metrics = compute_metrics(val_logits_arr, val_fold_df["answer"].tolist())

        print(
            f"  Fold {fold} Epoch {epoch:02d} | Loss: {running_loss / len(train_loader):.4f} | Val MAP@3: {fold_metrics['map3']:.4f} | Val Acc: {fold_metrics['accuracy']:.4f}"
        )
        wandb.log(
            {
                f"fold_{fold}/map3": fold_metrics["map3"],
                f"fold_{fold}/loss": running_loss / len(train_loader),
            }
        )

        if fold_metrics["map3"] > best_val_map3:
            best_val_map3 = fold_metrics["map3"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f"  --> Best Val MAP@3 for Fold {fold}: {best_val_map3:.4f}")
    model.load_state_dict(best_state)
    model.eval()

    val_logits_best = []
    with torch.no_grad():
        for input_ids, attention_mask, _ in val_loader:
            val_logits_best.append(
                model(input_ids.to(DEVICE), attention_mask.to(DEVICE)).cpu().numpy()
            )
    train_oof_logits[val_idx] = np.concatenate(val_logits_best, axis=0)

    fold_test_logits = []
    with torch.no_grad():
        for input_ids, attention_mask in test_loader:
            fold_test_logits.append(
                model(input_ids.to(DEVICE), attention_mask.to(DEVICE)).cpu().numpy()
            )
    test_logits_matrix += np.concatenate(fold_test_logits, axis=0) / N_FOLDS

    torch.save(best_state, os.path.join(OUTPUT_DIR, f"electra_fold_{fold}.pt"))
    del model, optimizer, scheduler, train_loader, val_loader, best_state
    gc.collect()
    torch.cuda.empty_cache()

print("=" * 65)
oof_metrics = compute_metrics(train_oof_logits, train_df["answer"].tolist())
print(f"5-Fold Overall Out-of-Fold (OOF) MAP@3    : {oof_metrics['map3']:.4f}")
print(f"5-Fold Overall Out-of-Fold (OOF) Accuracy : {oof_metrics['accuracy']:.4f}")

wandb.log({"oof/map3": oof_metrics["map3"], "oof/accuracy": oof_metrics["accuracy"]})
wandb.summary["best_oof_map3"] = oof_metrics["map3"]

## 7. Generate Kaggle Test Predictions & Save Submission

In [ ]:
test_preds_top3 = [
    " ".join([OPTIONS[i] for i in np.argsort(-test_logits_matrix[k])[:3]])
    for k in range(len(test_logits_matrix))
]

submission_df = pd.DataFrame(
    {
        "id": test_df["id"].values,
        "Prediction": test_preds_top3,
    }
)

SUBMISSION_PATH = "../Submissions/submission_electra_5fold.csv"
os.makedirs("../Submissions", exist_ok=True)
submission_df.to_csv(SUBMISSION_PATH, index=False)
submission_df.to_csv("submission.csv", index=False)

print("=" * 60)
print(submission_df.head(10).to_string(index=False))
print("=" * 60)
print(f"5-Fold Submission saved → {SUBMISSION_PATH} & submission.csv ✓")

## 8. Out-of-Fold Error Analysis

In [ ]:
oof_analysis_df = train_df.copy()
oof_preds_top3 = [
    [OPTIONS[i] for i in np.argsort(-train_oof_logits[k])[:3]]
    for k in range(len(train_oof_logits))
]

oof_analysis_df["pred_top1"] = [p[0] for p in oof_preds_top3]
oof_analysis_df["pred_top3"] = [" ".join(p) for p in oof_preds_top3]
oof_analysis_df["correct"] = oof_analysis_df["pred_top1"] == oof_analysis_df["answer"]
oof_analysis_df["in_top3"] = [
    a in p for a, p in zip(oof_analysis_df["answer"], oof_preds_top3)
]

errors = oof_analysis_df[~oof_analysis_df["correct"]]
not_in_top3 = oof_analysis_df[~oof_analysis_df["in_top3"]]

print(
    f"OOF Top-1 Errors : {len(errors)}/{len(oof_analysis_df)} = {len(errors) / len(oof_analysis_df) * 100:.1f}%"
)
print(
    f"OOF Not Top-3    : {len(not_in_top3)}/{len(oof_analysis_df)} = {len(not_in_top3) / len(oof_analysis_df) * 100:.1f}%"
)

for _, row in errors.head(5).iterrows():
    print(f"\nQ  : {row['prompt'][:100]}...")
    print(f"True : {row['answer']}   |   Pred Top-3 : {row['pred_top3']}")

## 9. Finish W&B Experiment Run

In [ ]:
wandb.finish()
print("W&B experiment tracking completed ✓")